## Buisness Transaction Dataset Cleaning

**Author:** Sunmi  
**Dataset:** Buisness Transaction Dataset (213 initial records)

---

## Executive Summary

This notebook documents the data cleaning process for a Buisness Transacton dataset. The dataset contains information about 213 transactions.

**Key Cleaning Operations:**
- Removed 12 duplicate customer records
- Handled Date column 
- Handled missing values by using statistical imputation
- Handled missing values by dropping column with too many missing values
- Handled some columns containing distorted values,abbrevated and mis-spelt words  

- Preserved data integrity while maintaining business logic

**Final Output:** A clean dataset of 198 unique transactions ready for exploratory data analysis

---
## 1. Environment Setup

We begin by importing the necessary Python libraries for data manipulation and analysis. Warnings are suppressed to maintain clean output throughout the notebook.

In [1]:
import numpy as np
import pandas as pd
import warnings
import re
warnings.filterwarnings('ignore')

---
## 2. Data Loading & Initial Inspection

The raw dataset is loaded from CSV format. This initial inspection helps us understand the data structure, identify potential quality issues, and plan our cleaning strategy.


In [2]:
df=pd.read_csv(r"c:\Users\HP USER\Downloads\messy_financial_data.csv")

In [3]:
df.columns

Index(['Transaction_ID', 'Date', 'customer name', 'CUSTOMER_ID', 'Product',
       'Category', 'Amount', 'Currency', 'Exchange_Rate', 'Amount_USD',
       'Payment_Method', 'Status', 'Region', 'Sales_Rep', 'Commission%', 'Tax',
       'Discount', 'NET AMOUNT', 'Invoice#', 'Notes'],
      dtype='object')

In [4]:
df.shape

(213, 20)

The dataset contains **213 rows** (customer records) and **20 columns** (features). This is our starting point before any cleaning operations.

In [5]:
df.head(20)

,Transaction_ID,Date,customer name,CUSTOMER_ID,Product,Category,Amount,Currency,Exchange_Rate,Amount_USD,Payment_Method,Status,Region,Sales_Rep,Commission%,Tax,Discount,NET AMOUNT,Invoice#,Notes
0,0169,2023-07-13,ACME CORP.,C003,data storage,products,29462,EUR,1.08,31818.96,credit card,pending,NORTH AMERICA,Carlos Mendez,0.0392,4419.3000,0.07,31818.96,32566,NaN
1,txn0018,06-22-2023,JOHN SMITH,C001,Training,Services,3297,CAD,0.74,2439.78,Cheque,Cancelled,Europe,TBD,10.8%,263.7600,0.12,3165.12,Invoice-15096,disputed
2,TXN-0118,01/04/2023,Tech Solutions,C005,Data Storage,Products,11981,GBP,1.27,15215.87,Check,COMPLETED,Asia Pacific,NaN,0.1347,0.0000,0.03,11621.57,INV31178,NaN
3,TXN0190,24-03-2023,Maria Garcia,C007,Training,Services,15901,USD,1,15901.00,CC,Complete,EU,NaN,0.0763,1590.1000,0.13,15423.97,INV-45941,DISPUTED
4,TXN-0025,2023/07/03,XYZ Enterprises,NaN,Consulting,Services,48941.73,GBP,1.27,62156.00,Check,Cancelled,NORTH AMERICA,mike wilson,0.0643,9788.3500,0.22,47962.90,95816,NaN
5,TXN #0061,2023-07-05,Sarah Connor,C006,hardware,Products,6765,GBP,1.27,8591.55,wire transfer,Complete,APAC,Carlos Mendez,0.1133,541.2000,0.22,5817.90,64574,Recurring
6,txn0133,07/06/2023,john smith,C001,hardware,Products,744.84,GBP,1.27,945.95,CC,Complete,EUROPE,TBD,0.1212,59.5900,0.01,796.98,inv-87216,One-time
7,TXN0005,24/10/2023,John Smith,C001,Support Contract,Services,44054.14,GBP,1.27,55948.76,CHECK,CANCELLED,Europe,Tom Brown,0.1126,4405.4100,0.05,46256.84,Invoice-47347,Follow up needed
8,txn0148,"July 08, 2023",Maria Garcia,C007,hardware,Products,19155,EUR,1.08,20687.40,Wire Transfer,CANCELLED,EU,NaN,0.1055,3831.0000,0.13,20495.85,inv-49438,Late payment
9,txn0006,09-19-2023,Jane Doe,C002,Hardware,Products,21150.1,USD,1,21150.10,credit card,Canceled,LATAM,Carlos Mendez,0.0865,NaN,0.07,19669.59,Invoice-73672,OK


**Initial Observations:**
- Distorted values are visible in `transaction ID` 
- `Transaction Date` column are not in properly aligned format
- Inconsistent casing are visible across the columns
- Missing values are visible across the columns
- Some rows are shifted,Dates appearing in the customer name column is the telltale sign 

### 3. Removing Shifted Rows

Lets check out the rows that shifted columns 

In [6]:
shifted_mask = df['customer name'].str.match(r'\d{1,4}[-/\.]\d{1,2}[-/\.]\d{1,4}', na=False)

df[shifted_mask]

,Transaction_ID,Date,customer name,CUSTOMER_ID,Product,Category,Amount,Currency,Exchange_Rate,Amount_USD,Payment_Method,Status,Region,Sales_Rep,Commission%,Tax,Discount,NET AMOUNT,Invoice#,Notes
14,NaN,TXN-0110,10-01-2023,xyz enterprises,C008,data storage,products,-34660.7,USD,1.00,-34660.7,Cash,CANCELLED,Asia Pacific,LISA CHEN,0.1244,-1733.03,0.15,NaN,INV76722
94,NaN,txn0177,12/13/2023,JOHN SMITH,C001,Data Storage,Products,30561,EUR,1.08,33005.88,credit card,Complete,NORTH AMERICA,Tom Brown,0.0394,1528.05,0.08,29644.17,Invoice-55596
142,NaN,TXN-0111,04/26/2023,Sarah Connor,NaN,Cloud Subscription,Software,12452,GBP,1.27,15814.04,PAYPAL,Completed,Asia Pacific,Carlos Mendez,0.1178,0,0.04,11953.92,INV50607


In [7]:
df = df[~shifted_mask]

print(f"Rows removed: {shifted_mask.sum()}")
print(f"Remaining rows: {len(df)}")

Rows removed: 3
Remaining rows: 210


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 210 entries, 0 to 212
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Transaction_ID  205 non-null    object 
 1   Date            201 non-null    object 
 2   customer name   205 non-null    object 
 3   CUSTOMER_ID     196 non-null    object 
 4   Product         205 non-null    object 
 5   Category        205 non-null    object 
 6   Amount          204 non-null    object 
 7   Currency        205 non-null    object 
 8   Exchange_Rate   197 non-null    object 
 9   Amount_USD      186 non-null    float64
 10  Payment_Method  205 non-null    object 
 11  Status          205 non-null    object 
 12  Region          205 non-null    object 
 13  Sales_Rep       166 non-null    object 
 14  Commission%     202 non-null    object 
 15  Tax             188 non-null    float64
 16  Discount        205 non-null    object 
 17  NET AMOUNT      187 non-null    float64


## 4. Handling Duplicate Records

Duplicate Transaction records can skew our analysis . We identify and remove exact duplicates across all columns to ensure each one appears only once in our dataset.

In [9]:
df.duplicated().sum()

np.int64(12)

### Lets take a look at the duplicates records in our dataset

In [10]:
df[df.duplicated()]

,Transaction_ID,Date,customer name,CUSTOMER_ID,Product,Category,Amount,Currency,Exchange_Rate,Amount_USD,Payment_Method,Status,Region,Sales_Rep,Commission%,Tax,Discount,NET AMOUNT,Invoice#,Notes
72,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86,txn0053,2023-10-21,Tech Solutions Ltd,C005,Training,Services,38144.0125,JPY,0.0067,255.56,CHECK,Cancelled,N. America,mike wilson,0.0909,5721.60,0.06,41576.97,Invoice-15400,NaN
102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
111,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
125,txn0018,06-22-2023,JOHN SMITH,C001,Training,Services,3297,CAD,0.74,2439.78,Cheque,Cancelled,Europe,TBD,10.8%,263.76,0.12,3165.12,Invoice-15096,disputed
144,txn0069,2023/07/17,john smith,C001,Software license,Software,10133,EUR,1.08,10943.64,Credit Card,Complete,N. America,Lisa Chen,0.1153,810.64,0.09,10031.67,#INV-29865,NaN
160,txn0029,15/09/2023,JOHN SMITH,C001,data storage,products,1101,CAD,0.74,814.74,ACH,Failed,LATAM,LISA CHEN,0.0639,110.10,0.25,935.85,79009,Credit memo
172,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
179,txn0069,2023/07/17,john smith,C001,Software license,Software,10133,EUR,1.08,10943.64,Credit Card,Complete,N. America,Lisa Chen,0.1153,810.64,0.09,10031.67,#INV-29865,NaN
205,TXN0072,2023-05-19,John Smith,C001,support contract,services,11043.4633,GBP,1.27,14025.20,PAYPAL,PENDING,North America,TBD,0.0542,0.00,0.12,9718.25,INV94368,DISPUTED


In [11]:
df.drop_duplicates(inplace=True)

In [12]:
df.duplicated().sum()

np.int64(0)

**Result:** The dataset now contains **202 unique records** (reduced from 218), indicating that **12 duplicate rows** were successfully removed.

### Fixing the inconsistencies in the date column

In [13]:
# Check unique values (sample)
print(df['Date'].unique()[:10])

['2023-07-13' '06-22-2023' '01/04/2023' '24-03-2023' '2023/07/03'
 '2023-07-05' '07/06/2023' '24/10/2023' 'July 08, 2023' '09-19-2023']


In [14]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

In [15]:
df.head(20)

,Transaction_ID,Date,customer name,CUSTOMER_ID,Product,Category,Amount,Currency,Exchange_Rate,Amount_USD,Payment_Method,Status,Region,Sales_Rep,Commission%,Tax,Discount,NET AMOUNT,Invoice#,Notes
0,0169,2023-07-13,ACME CORP.,C003,data storage,products,29462,EUR,1.08,31818.96,credit card,pending,NORTH AMERICA,Carlos Mendez,0.0392,4419.30,0.07,31818.96,32566,NaN
1,txn0018,NaT,JOHN SMITH,C001,Training,Services,3297,CAD,0.74,2439.78,Cheque,Cancelled,Europe,TBD,10.8%,263.76,0.12,3165.12,Invoice-15096,disputed
2,TXN-0118,NaT,Tech Solutions,C005,Data Storage,Products,11981,GBP,1.27,15215.87,Check,COMPLETED,Asia Pacific,NaN,0.1347,0.00,0.03,11621.57,INV31178,NaN
3,TXN0190,NaT,Maria Garcia,C007,Training,Services,15901,USD,1,15901.00,CC,Complete,EU,NaN,0.0763,1590.10,0.13,15423.97,INV-45941,DISPUTED
4,TXN-0025,NaT,XYZ Enterprises,NaN,Consulting,Services,48941.73,GBP,1.27,62156.00,Check,Cancelled,NORTH AMERICA,mike wilson,0.0643,9788.35,0.22,47962.90,95816,NaN
5,TXN #0061,2023-07-05,Sarah Connor,C006,hardware,Products,6765,GBP,1.27,8591.55,wire transfer,Complete,APAC,Carlos Mendez,0.1133,541.20,0.22,5817.90,64574,Recurring
6,txn0133,NaT,john smith,C001,hardware,Products,744.84,GBP,1.27,945.95,CC,Complete,EUROPE,TBD,0.1212,59.59,0.01,796.98,inv-87216,One-time
7,TXN0005,NaT,John Smith,C001,Support Contract,Services,44054.14,GBP,1.27,55948.76,CHECK,CANCELLED,Europe,Tom Brown,0.1126,4405.41,0.05,46256.84,Invoice-47347,Follow up needed
8,txn0148,NaT,Maria Garcia,C007,hardware,Products,19155,EUR,1.08,20687.40,Wire Transfer,CANCELLED,EU,NaN,0.1055,3831.00,0.13,20495.85,inv-49438,Late payment
9,txn0006,NaT,Jane Doe,C002,Hardware,Products,21150.1,USD,1,21150.10,credit card,Canceled,LATAM,Carlos Mendez,0.0865,NaN,0.07,19669.59,Invoice-73672,OK


In [16]:
df['Date']= pd.to_datetime(df['Date'], infer_datetime_format=True, errors='coerce')

In [17]:
df['Date']= pd.to_datetime(df['Date'], dayfirst=False, errors='coerce')

In [18]:
mask = df['Date'].isna()
print(f"Still unparsed: {mask.sum()} rows")

Still unparsed: 171 rows


This shows that the number of NaT(date rows that failed conversion) dominates the column and if left untouched can affect our analysis.
It is best for the date column to be dropped. 

In [19]:
df.drop(columns=['Date'], inplace=True)

The date column has been successfully dropped.

---
## 6. Missing Value Analysis

Let's examine the state of the dataset more closely. Understanding the distribution and characteristics of our features helps inform the best approach for handling missing values.

In [20]:
df.isna().sum()

Transaction_ID     1
customer name      1
CUSTOMER_ID       10
Product            1
Category           1
Amount             2
Currency           1
Exchange_Rate      9
Amount_USD        19
Payment_Method     1
Status             1
Region             1
Sales_Rep         40
Commission%        4
Tax               17
Discount           1
NET AMOUNT        18
Invoice#           1
Notes             61
dtype: int64

### Checking for percentage of  the missing values

In [21]:
df.isnull().sum()/len(df)*100

Transaction_ID     0.505051
customer name      0.505051
CUSTOMER_ID        5.050505
Product            0.505051
Category           0.505051
Amount             1.010101
Currency           0.505051
Exchange_Rate      4.545455
Amount_USD         9.595960
Payment_Method     0.505051
Status             0.505051
Region             0.505051
Sales_Rep         20.202020
Commission%        2.020202
Tax                8.585859
Discount           0.505051
NET AMOUNT         9.090909
Invoice#           0.505051
Notes             30.808081
dtype: float64

It is observed that:
- Notes column contains >30% null values
- The remaining columns contains <30%
thus, the notes column would be dropped and the remaining colunms would be filled.
The numerical columns would be filled with median and the categorical column would be filled with mode. 

Dropping the notes column 

In [22]:
df.drop(columns=['Notes'], inplace=True)

## 7.Imputation Strategy:

Given the minimal number of missing values in some other rows, we'll use statistical imputation methods that preserve the distribution characteristics of each feature:

1. **Numeric Features:** Use **median imputation** rather than mean to reduce the impact of potential outliers.
2. **Categorical Features:** Use **mode imputation** to fill.

This approach maintains data integrity while avoiding information loss from row deletion.

In [23]:
for col in df.select_dtypes(include='number'):
    df[col].fillna(df[col].median(), inplace=True)

for col in df.select_dtypes(include='object'):
    df[col].fillna(df[col].mode()[0], inplace=True)

Checking if all null values has been handled

In [24]:
df.isna().sum()

Transaction_ID    0
customer name     0
CUSTOMER_ID       0
Product           0
Category          0
Amount            0
Currency          0
Exchange_Rate     0
Amount_USD        0
Payment_Method    0
Status            0
Region            0
Sales_Rep         0
Commission%       0
Tax               0
Discount          0
NET AMOUNT        0
Invoice#          0
dtype: int64

All null values has been successfully handled

## 8. Distorted column analysis

In [25]:
df.head()

,Transaction_ID,customer name,CUSTOMER_ID,Product,Category,Amount,Currency,Exchange_Rate,Amount_USD,Payment_Method,Status,Region,Sales_Rep,Commission%,Tax,Discount,NET AMOUNT,Invoice#
0,0169,ACME CORP.,C003,data storage,products,29462,EUR,1.08,31818.96,credit card,pending,NORTH AMERICA,Carlos Mendez,0.0392,4419.30,0.07,31818.96,32566
1,txn0018,JOHN SMITH,C001,Training,Services,3297,CAD,0.74,2439.78,Cheque,Cancelled,Europe,TBD,10.8%,263.76,0.12,3165.12,Invoice-15096
2,TXN-0118,Tech Solutions,C005,Data Storage,Products,11981,GBP,1.27,15215.87,Check,COMPLETED,Asia Pacific,Lisa Chen,0.1347,0.00,0.03,11621.57,INV31178
3,TXN0190,Maria Garcia,C007,Training,Services,15901,USD,1,15901.00,CC,Complete,EU,Lisa Chen,0.0763,1590.10,0.13,15423.97,INV-45941
4,TXN-0025,XYZ Enterprises,C001,Consulting,Services,48941.73,GBP,1.27,62156.00,Check,Cancelled,NORTH AMERICA,mike wilson,0.0643,9788.35,0.22,47962.90,95816


looking deeper into this dataset,it is noticed that
- The `transaction-id` column has distorted values
- The `Invoice#` column contains distorted values
- The `amount` column contains some values with the $ sign and some without it

Let's put things in it's right place

Starting with the `transaction-id` and  `Invoice#` column

In [26]:
# Function to extract numeric part
def clean_id(value):
    if pd.isna(value):
        return value
    # Convert to string
    value = str(value)
    # Extract digits
    digits = re.findall(r'\d+', value)
    return ''.join(digits) if digits else None

# Apply to columns
df['Invoice#'] = df['Invoice#'].apply(clean_id)
df['Transaction_ID'] = df['Transaction_ID'].apply(clean_id)

Now onto the `Amount` column

In [27]:
df['Amount'] = (
    df['Amount']
    .astype(str)
    .str.replace(r'[\$,]', '', regex=True)
)

---
## 9. Data Type Standardization

After we convert inappropriate features to their appropriate numeical data types.

### Changing the incorrect data types

In [28]:
df['Discount'] = pd.to_numeric(df['Discount'],errors='coerce')
df['Commission%'] = pd.to_numeric(df["Commission%"],errors='coerce')
df['Amount'] = pd.to_numeric(df['Amount'],errors='coerce')
df['Exchange_Rate'] = pd.to_numeric(df['Exchange_Rate'],errors='coerce')
df['Invoice#'] = pd.to_numeric(df['Invoice#'], errors='coerce')
df['Transaction_ID'] = pd.to_numeric(df['Transaction_ID'], errors='coerce')
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')

An observation was made in the `Payment_Method` column and `Region` column,Some words in those columns are abbreviated and spelt wrongly.

Words like that if left unchanged can distort our analysis.
Let's fix that..

In [29]:
#we replace the abbreviated and mis-spelt words in the payment method column to the appropriate one.
df['Payment_Method'] = df['Payment_Method'].replace({
    'CC': 'Credit Card',
    'Check':'Cheque'
})

In [30]:
# We replace the abbreviated words in the Region column with the appropriate words
df['Region'] = df['Region'].replace({
    'EU': 'Europe',
    'APAC':'Asia Pacific',
    'LATAM':'Latin America',
    'N. America': 'North America'
})

The casing in the categorical columns are  quite unusual and can affect our analysis.
So we will adjust the casing so everything aligns properly 

In [31]:

text_cols = df.select_dtypes(include='object').columns
df[text_cols] = df[text_cols].apply(lambda x: x.str.strip().str.title())

In [32]:
df['Currency'] = df['Currency'].str.upper().str.strip()

In [33]:
df.head()

,Transaction_ID,customer name,CUSTOMER_ID,Product,Category,Amount,Currency,Exchange_Rate,Amount_USD,Payment_Method,Status,Region,Sales_Rep,Commission%,Tax,Discount,NET AMOUNT,Invoice#
0,169,Acme Corp.,C003,Data Storage,Products,29462.00,EUR,1.08,31818.96,Credit Card,Pending,North America,Carlos Mendez,0.0392,4419.30,0.07,31818.96,32566
1,18,John Smith,C001,Training,Services,3297.00,CAD,0.74,2439.78,Cheque,Cancelled,Europe,Tbd,NaN,263.76,0.12,3165.12,15096
2,118,Tech Solutions,C005,Data Storage,Products,11981.00,GBP,1.27,15215.87,Cheque,Completed,Asia Pacific,Lisa Chen,0.1347,0.00,0.03,11621.57,31178
3,190,Maria Garcia,C007,Training,Services,15901.00,USD,1.00,15901.00,Credit Card,Complete,Europe,Lisa Chen,0.0763,1590.10,0.13,15423.97,45941
4,25,Xyz Enterprises,C001,Consulting,Services,48941.73,GBP,1.27,62156.00,Cheque,Cancelled,North America,Mike Wilson,0.0643,9788.35,0.22,47962.90,95816


Some missing values were noticed in the `Commission%` and `Discount`,lets see how much missing values the columns contain 

In [34]:
df.isna().sum()

Transaction_ID     0
customer name      0
CUSTOMER_ID        0
Product            0
Category           0
Amount             0
Currency           0
Exchange_Rate      0
Amount_USD         0
Payment_Method     0
Status             0
Region             0
Sales_Rep          0
Commission%       12
Tax                0
Discount          14
NET AMOUNT         0
Invoice#           0
dtype: int64

Both columns contains minimum missing values.
We would fill them with median.

In [35]:
for col in df.select_dtypes(include='number'):
    df[col].fillna(df[col].median(), inplace=True)


The columns has been successfully filled.

## 5. Final Dataset Validation

Let's verify that our cleaning operations were successful by examining the final state of the dataset.
 
 Cross checking the dataset for any missing or duplicate values

In [36]:
df.isna().sum()

Transaction_ID    0
customer name     0
CUSTOMER_ID       0
Product           0
Category          0
Amount            0
Currency          0
Exchange_Rate     0
Amount_USD        0
Payment_Method    0
Status            0
Region            0
Sales_Rep         0
Commission%       0
Tax               0
Discount          0
NET AMOUNT        0
Invoice#          0
dtype: int64

In [37]:
df.duplicated().sum()

np.int64(0)

In [38]:
df.shape

(198, 18)

### Save the already clean dataset

In [39]:
df.to_csv("clean_buisness_transaction_dataset")

**Output:** `clean_buisness_transaction_dataset`  
**Records:** 198 unique rows  
**Features:** 18 columns (all complete, no missing values)